# Imports

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [16]:
level = '3'

In [17]:
file_codes = ['ADNIMERGE']#, 'UCSFFSX', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL'] #, 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 'BLCHANGE', 'DXSUM', 

In [18]:
search = client.query_files(
    query={'custom.level' : 'cleaned_0'+level, 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [19]:
print(zip_files.keys())

dict_keys(['ADNIMERGE_25Jul2025_03.csv'])


# Import support file already populated

In [20]:
support_file = pd.read_excel('ADNI_variables_cleaned'+ level +'.xlsx')
dataCleaner = DataCleaner(support_file=support_file)

# Rename Varibles

In [21]:
for file_name in zip_files.keys():
    print('\n------------------------- ', file_name)
    dataset = zip_files[file_name]

    df_new = dataset.copy(deep=True) 

    df_new.rename(columns={'APOE4': 'APOE_4'}, inplace=True) #, 'RVentricle%ICV': 'RVentricles%ICV', 'LVentricle%ICV': 'LVentricles%ICV'}, inplace=True)
    display(df_new.head())

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(df_new, new_level='cleaned_0'+level, file_name=file_name, prefix='cleaned/single_file', updated_support_file=support_file)  

    result = client.upload_dataframe(
        df=df_new,
        object_name=file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )



-------------------------  ADNIMERGE_25Jul2025_03.csv


,AGE_bl,VISIT_MONTH,PTID,RID,COHORT,EXAMDATE,AGE,EDUCAT,APOE_4,CDRSB,...,RACE/White,DX/CN,DX/Dementia,DX/MCI,Ventricles%ICV,Hippocampus%ICV,Entorhinal%ICV,Fusiform%ICV,MidTemp%ICV,ICV%ICV
0,74.3,0,011_S_0002,2,ADNI1,2005-09-08,74.3,16,0.0,0.0,...,True,True,False,False,5.957343,0.420022,0.210464,0.834349,1.407596,100.0
1,74.3,6,011_S_0002,2,ADNI1,2006-03-06,74.7,16,0.0,0.0,...,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN
2,74.3,36,011_S_0002,2,ADNI1,2008-08-27,77.2,16,0.0,0.0,...,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN
3,74.3,60,011_S_0002,2,ADNIGO,2010-09-22,79.3,16,0.0,0.0,...,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN
4,74.3,72,011_S_0002,2,ADNI2,2011-09-19,80.3,16,0.0,0.0,...,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
for file_name in zip_files.keys():
    metadata = client.get_metadata(object_name='cleaned/single_file/'+file_name)
    print('-----------------', file_name, '\n',metadata['metadata']['custom'])

----------------- ADNIMERGE_25Jul2025_03.csv 
 {'cofattori': ['EDUCAT', 'APOE_4', 'GENDER/female', 'GENDER/male', 'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed', 'ETHNICITY/latino', 'ETHNICITY/not_latino', 'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american', 'RACE/White'], 'file_code': 'ADNIMERGE', 'level': 'cleaned_03', 'norm_intervallo': [], 'norm_scala': ['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ'], 'norm_scale_value': {'ADAS11': [0, 70, 'increasing'], 'ADAS13': [0, 70, 'increasing'], 'CDRSB': [0, 18, 'increasing'], 'FAQ': [0, 30, 'increasing'], 'MMSE': [0, 30, 'inverse'], 'RAVLT_immediate': [0, 75, 'inverse']}, 'norm_volume': ['Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV', 'ICV%ICV'], 'population': ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3'], 'predittori': ['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%